**Model Application:**

- Load a pre-trained sentiment analysis model from Hugging Face Transformers.
- Apply the model to a subset of the chosen dataset (e.g., the first 1000 samples from the training set).
- Evaluate the model's performance. You can start with qualitative analysis (inspecting predictions) and then explore quantitative metrics.

In [19]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from datasets import load_dataset

# Load the Rotten Tomatoes dataset
dataset = load_dataset("rotten_tomatoes")

# Print the dataset information
print(dataset)

# Example: Accessing the training split
train_dataset = dataset["train"]

# Print the first example in the training set
print(train_dataset[0])

In [ ]:
# output from above cell:
Generating train split: 100%|██████████| 8530/8530 [00:00<00:00, 193571.39 examples/s]
Generating validation split: 100%|██████████| 1066/1066 [00:00<00:00, 207226.92 examples/s]
Generating test split: 100%|██████████| 1066/1066 [00:00<00:00, 226834.16 examples/s]
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})
{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}


In [ ]:
# Select first 1000 samples from the 'train' split
#subset_train = dataset['train'].select(range(1000)) # all positive
test_ds = dataset['test']

# view example
print(test_ds[7])  # {text, label}
print(test_ds[7]['text'])
print(test_ds[7]['label'])
len(test_ds['text'])


{'text': 'weighty and ponderous but every bit as filling as the treat of the title .', 'label': 1}
weighty and ponderous but every bit as filling as the treat of the title .
1


1066

In [ ]:
# ref https://huggingface.co/docs/transformers/en/main_classes/pipelines
# https://huggingface.co/docs/transformers/main/en/task_summary

from transformers import pipeline

# # Loads the default sentiment analysis model
# classifier = pipeline("sentiment-analysis")

# To use a specific model from the Hub, specify the 'model' parameter:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

example = classifier("I love using Hugging Face models!")
print(example) 
# Output: [{'label': 'POSITIVE', 'score': 0.99}]


In [ ]:
# Using TEST dataset, iterate through it to get predictions
test_preds = classifier([i['text'] for i in test_ds])

In [32]:
# how preds look like
test_preds[0], test_preds[999]

({'label': 'POSITIVE', 'score': 0.9998145699501038},
 {'label': 'NEGATIVE', 'score': 0.9997528195381165})

In [34]:
# convert predictions to numeric labels (y_pred)
pred_labels = [1 if p['label'] == 'POSITIVE' else 0 for p in test_preds]
pred_labels[0], pred_labels[999]

(1, 0)

In [35]:
# get true labels (y_true)
true_labels = [i['label'] for i in test_ds]
true_labels[0], true_labels[999] 

(1, 0)

In [37]:
# eval model's performance using metrics
from sklearn.metrics import classification_report

print(classification_report(true_labels, pred_labels, target_names=["NEGATIVE", "POSITIVE"]))

              precision    recall  f1-score   support

    NEGATIVE       0.89      0.90      0.90       533
    POSITIVE       0.90      0.89      0.90       533

    accuracy                           0.90      1066
   macro avg       0.90      0.90      0.90      1066
weighted avg       0.90      0.90      0.90      1066



### Given solution

In [ ]:
from transformers import pipeline
from datasets import load_dataset
import pandas as pd

# Load the Rotten Tomatoes dataset
dataset = load_dataset("rotten_tomatoes")
train_dataset = dataset["train"]

# Load a pre-trained sentiment analysis model
sentiment_analysis = pipeline("sentiment-analysis")

# Select a subset of the dataset (e.g., 1000 samples)
subset_size = 1000
subset_dataset = train_dataset.select(range(subset_size))

# Make predictions on the subset
predictions = sentiment_analysis([example["text"] for example in subset_dataset])

# Add predictions to the dataset
df = pd.DataFrame(subset_dataset)
df['sentiment'] = [prediction['label'] for prediction in predictions]
df['sentiment_score'] = [prediction['score'] for prediction in predictions]

print(df.head())

# Example of qualitative analysis: Inspecting predictions
for i in range(5):
    print(f"Review: {subset_dataset[i]['text']}")
    print(f"Predicted Sentiment: {predictions[i]['label']} (Score: {predictions[i]['score']:.4f})\n")

# Further steps: Quantitative analysis (e.g., accuracy, F1-score) would require ground truth labels
# and a defined evaluation metric.  This is more complex because the Rotten Tomatoes dataset
# provides 'label' as 0 or 1, while the sentiment analysis pipeline provides 'POSITIVE' or 'NEGATIVE'.
# A mapping would be required to compare the two.